In [5]:
import numpy as np
import pandas as pd
from math import log, sqrt, exp
from scipy.stats import norm

In [6]:
def gbsm_european_with_greeks(option_type, S, K, T, r, q, sigma):
    d1 = (log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * sqrt(T))
    d2 = d1 - sigma * sqrt(T)

    if option_type.lower() == "call":
        value = S * exp(-q * T) * norm.cdf(d1) - K * exp(-r * T) * norm.cdf(d2)
        delta = exp(-q * T) * norm.cdf(d1)
        rho = K * T * exp(-r * T) * norm.cdf(d2)
        theta = (
            -S * exp(-q * T) * norm.pdf(d1) * sigma / (2 * sqrt(T))
            - r * K * exp(-r * T) * norm.cdf(d2)
            + q * S * exp(-q * T) * norm.cdf(d1)
        )
    else:            # Put option
        value = K * exp(-r * T) * norm.cdf(-d2) - S * exp(-q * T) * norm.cdf(-d1)
        delta = exp(-q * T) * (norm.cdf(d1) - 1)
        rho = -K * T * exp(-r * T) * norm.cdf(-d2)
        theta = (
            -S * exp(-q * T) * norm.pdf(d1) * sigma / (2 * sqrt(T))
            + r * K * exp(-r * T) * norm.cdf(-d2)
            - q * S * exp(-q * T) * norm.cdf(-d1)
        )

    gamma = exp(-q * T) * norm.pdf(d1) / (S * sigma * sqrt(T))
    vega = S * exp(-q * T) * norm.pdf(d1) * sqrt(T)

    return value, delta, gamma, vega, rho, theta

df = pd.read_csv("/Users/fuyuxuan/Downloads/test12_1.csv")

# Remove blank rows
df = df.dropna(how="all").copy()

results = []

for _, row in df.iterrows():
    option_id = int(row["ID"])
    option_type = row["Option Type"]
    S = float(row["Underlying"])
    K = float(row["Strike"])
    T = float(row["DaysToMaturity"]) / float(row["DayPerYear"])
    r = float(row["RiskFreeRate"])
    q = float(row["DividendRate"])
    sigma = float(row["ImpliedVol"])

    value, delta, gamma, vega, rho, theta = gbsm_european_with_greeks(option_type, S, K, T, r, q, sigma)
    results.append([option_id, value, delta, gamma, vega, rho, theta])

out = pd.DataFrame(results, columns=["ID", "Value", "Delta", "Gamma", "Vega", "Rho", "Theta"])
print(out)

   ID      Value     Delta     Gamma       Vega        Rho      Theta
0   1   3.260824  0.547872  0.053506  14.659079   7.058414 -13.019817
1   2   2.646281 -0.452128  0.053506  14.659079  -6.556032  -8.547471
2   3  22.043329  0.685508  0.008115  35.571438  50.967099  -7.213608
3   4  20.449083 -0.470751  0.009310  40.812329 -73.999110  -5.351164
